# QSVM estilo Farooq — PM2.5 extremos (t+24h)

Notebook **independente** para Kaggle, com:
- Features Farooq (`pm25/temperature` × min/max/median/variance)
- Clássicos + QSVM (PCA→2 + MinMax + ZZFeatureMap)
- Painel visual (Accuracy, F1, AUPRC, confusão, PR/ROC, learning curve)
- Protocolo **multi-seed** (média ± std + Wilcoxon)

**Settings Kaggle:** Internet **ON** · Accelerator **None** (CPU)

Escolha o tamanho do experimento na célula de config: `small`, `medium` ou `large`.


## 1. Instalação

In [ ]:
%pip install -q qiskit qiskit-machine-learning qiskit-aer scikit-learn pandas numpy matplotlib seaborn joblib scipy

## 2. Configuração + estimativa de tempo

| Modo | Sample | Seeds | Variantes QSVM | Tempo estimado (CPU Kaggle) |
|---|---:|---:|---:|---:|
| `small` | 80/40/40 | 1 | 2 | **~1–3 min** |
| `medium` | 200/60/80 | 5 | 2 | **~20–40 min** |
| `large` | 500/150/200 | 10 | 2 | **~6–12 h** |

O tempo é dominado pelo **kernel quântico** ∝ (n_train² × n_qubits × reps).


In [ ]:
from pathlib import Path
import math

ON_KAGGLE = Path('/kaggle/working').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('artifacts/kaggle_farooq_nb')
WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / 'data'
PLOTS = WORK / 'plots'
DATA.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)

# ==========================================================
# MODO: 'small' | 'medium' | 'large'
# ==========================================================
MODE = 'medium'

PRESETS = {
    'small': dict(train=80, val=40, test=40, seeds=[42], q_variants='both'),
    'medium': dict(train=200, val=60, test=80, seeds=[42, 7, 11, 13, 21], q_variants='both'),
    'large': dict(
        train=500, val=150, test=200,
        seeds=[42, 7, 11, 13, 21, 29, 37, 41, 47, 53],
        q_variants='both',
    ),
}

cfg_mode = PRESETS[MODE]
TRAIN_SIZE = cfg_mode['train']
VAL_SIZE = cfg_mode['val']
TEST_SIZE = cfg_mode['test']
SEEDS = list(cfg_mode['seeds'])
PRIMARY_SEED = SEEDS[0]  # usado no painel visual detalhado

STATION = 'Aotizhongxin'
HORIZON_HOURS = 24
EXTREME_PERCENTILE = 0.90
ROLL_WINDOW = 24
ROLL_MIN_PERIODS = 12
PCA_QUBITS = 2

# Variantes QSVM: (id, angular, reps)
ALL_Q_VARIANTS = [
    ('Q_farooq_pipeline', 'minmax_0_1', 1),
    ('Q_farooq_0pi', 'minmax_0_pi', 2),
]
if cfg_mode['q_variants'] == 'farooq_only':
    Q_VARIANTS = [ALL_Q_VARIANTS[0]]
else:
    Q_VARIANTS = ALL_Q_VARIANTS

# Estimativa empírica (CPU): ~0.0023 s por par no kernel 2 qubits
def estimate_minutes(n_train, n_test, n_seeds, variants):
    classical = 0.05 * n_seeds
    q = 0.0
    for _, _, reps in variants:
        pairs = n_train * n_train + n_test * n_train
        sec = pairs * 0.0023 * (1.0 + 0.1 * (reps - 1))
        q += sec * n_seeds / 60.0
    plots = 1.5 if n_seeds == 1 else 3.0
    return classical + q + plots

est = estimate_minutes(TRAIN_SIZE, TEST_SIZE, len(SEEDS), Q_VARIANTS)
print('=' * 60)
print(f'MODE={MODE}')
print(f'sample train/val/test = {TRAIN_SIZE}/{VAL_SIZE}/{TEST_SIZE}')
print(f'seeds ({len(SEEDS)}): {SEEDS}')
print(f'QSVM variants: {[v[0] for v in Q_VARIANTS]}')
print(f'Estimativa de tempo: ~{est:.0f} min  (faixa típica {est*0.7:.0f}–{est*1.4:.0f} min)')
if MODE == 'large':
    print('Atenção: modo large pode levar várias horas no Kaggle CPU.')
print('=' * 60)
print('WORK=', WORK)


## 3. Download dos dados (UCI Beijing)

In [ ]:
import zipfile
from io import BytesIO
from urllib.request import urlopen
import pandas as pd

UCI_URLS = [
    'https://archive.ics.uci.edu/ml/machine-learning-databases/00501/PRSA2017_Data_20130301-20170228.zip',
    'https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip',
]

csv_path = DATA / f'{STATION.lower()}_air_quality.csv'

if csv_path.exists():
    print('Usando cache:', csv_path)
    df_raw = pd.read_csv(csv_path)
else:
    last_err = None
    zip_bytes = None
    for url in UCI_URLS:
        try:
            print('Baixando', url)
            with urlopen(url, timeout=180) as resp:
                zip_bytes = resp.read()
            if zip_bytes:
                break
        except Exception as e:
            last_err = e
            print('falhou:', e)
    if not zip_bytes:
        raise RuntimeError(f'Download falhou: {last_err}')

    with zipfile.ZipFile(BytesIO(zip_bytes)) as zf:
        members = [n for n in zf.namelist() if STATION.lower() in n.lower() and n.lower().endswith('.csv')]
        if not members:
            raise FileNotFoundError(f'Sem CSV da estação {STATION}.')
        with zf.open(members[0]) as f:
            df_raw = pd.read_csv(f)
    df_raw.to_csv(csv_path, index=False)
    print('Salvo', csv_path, 'rows=', len(df_raw))

df_raw['timestamp'] = pd.to_datetime(dict(
    year=df_raw['year'], month=df_raw['month'], day=df_raw['day'], hour=df_raw['hour']
))
df_raw = df_raw.sort_values('timestamp').reset_index(drop=True)
print(df_raw[['timestamp', 'PM2.5', 'TEMP']].head(3))
print('período:', df_raw['timestamp'].min(), '→', df_raw['timestamp'].max())

## 4. Features estilo Farooq + split temporal (fixo para todas as seeds)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

np.random.seed(PRIMARY_SEED)
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

df = df_raw.copy()
for c in ['PM2.5', 'TEMP']:
    df[c] = df[c].ffill(limit=3)

source_map = {'PM2.5': 'pm25', 'TEMP': 'temperature'}
feat_cols = []
for src, prefix in source_map.items():
    roll = df[src].rolling(window=ROLL_WINDOW, min_periods=ROLL_MIN_PERIODS)
    df[f'{prefix}_min'] = roll.min()
    df[f'{prefix}_max'] = roll.max()
    df[f'{prefix}_median'] = roll.median()
    df[f'{prefix}_variance'] = roll.var()
    feat_cols += [f'{prefix}_min', f'{prefix}_max', f'{prefix}_median', f'{prefix}_variance']

df['future_pm25'] = df['PM2.5'].shift(-HORIZON_HOURS)
df = df.dropna(subset=['future_pm25'] + feat_cols).reset_index(drop=True)

n = len(df)
i_tr = int(n * 0.60)
i_va = i_tr + int(n * 0.20)
train_full = df.iloc[:i_tr].copy()
val_full = df.iloc[i_tr:i_va].copy()
test_full = df.iloc[i_va:].copy()

medians = train_full[feat_cols].median()
for part in (train_full, val_full, test_full):
    part[feat_cols] = part[feat_cols].fillna(medians)

threshold = float(train_full['future_pm25'].quantile(EXTREME_PERCENTILE))
for part in (train_full, val_full, test_full):
    part['target'] = (part['future_pm25'] >= threshold).astype(int)

print('features:', feat_cols)
print('threshold P90:', threshold)
print('sizes full:', len(train_full), len(val_full), len(test_full))
print('pos rate train/val/test:',
      float(train_full['target'].mean()),
      float(val_full['target'].mean()),
      float(test_full['target'].mean()))


def stratified_subsample(part, size, seed):
    size = min(size, len(part))
    if size < len(part):
        idx, _ = train_test_split(
            part.index, train_size=size, stratify=part['target'], random_state=seed
        )
        out = part.loc[idx].sort_values('timestamp').copy()
    else:
        out = part.copy()
    return out.reset_index(drop=True)

# Dataset processado completo (features Farooq + alvo) — útil para experimento/reprodução
feat_export_cols = ['timestamp'] + feat_cols + ['PM2.5', 'TEMP', 'future_pm25', 'target']
parts = []
for name, part in [('train', train_full), ('validation', val_full), ('test', test_full)]:
    tmp = part[feat_export_cols].copy()
    tmp['partition'] = name
    parts.append(tmp)
features_processed = pd.concat(parts, ignore_index=True)
features_path = WORK / 'features_processed.csv'
features_processed.to_csv(features_path, index=False)
print('Salvo', features_path, 'rows=', len(features_processed), 'cols=', list(features_processed.columns))


## 5. Funções de treino/avaliação (clássico + QSVM)

O split temporal é fixo; cada seed só muda a **subamostra estratificada** (protocolo reprodutível).

In [ ]:
import time
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score,
)
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.kernels import FidelityQuantumKernel

try:
    from qiskit_machine_learning.state_fidelities import ComputeUncompute
    _HAS_CU = True
except Exception:
    _HAS_CU = False


def proba_pos(model, X):
    if hasattr(model, 'predict_proba'):
        p = model.predict_proba(X)
        if hasattr(model, 'classes_') and 1 in list(model.classes_):
            return p[:, list(model.classes_).index(1)]
        return p[:, -1]
    if hasattr(model, 'decision_function'):
        s = np.asarray(model.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-s))
    return None


def metrics_row(y_true, y_hat, y_prob, name, **extra):
    row = {
        'model': name,
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'average_precision': float(average_precision_score(y_true, y_prob)) if y_prob is not None else np.nan,
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_hat)),
        'precision_extreme': float(precision_score(y_true, y_hat, zero_division=0)),
        'recall_extreme': float(recall_score(y_true, y_hat, zero_division=0)),
        'f1_extreme': float(f1_score(y_true, y_hat, zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_hat)),
        'auroc': float(roc_auc_score(y_true, y_prob)) if y_prob is not None else np.nan,
    }
    row.update(extra)
    return row


def make_kernel(n_qubits, reps=1):
    fmap = ZZFeatureMap(feature_dimension=n_qubits, reps=reps, entanglement='linear')
    if _HAS_CU:
        fidelity = ComputeUncompute(sampler=StatevectorSampler())
        return FidelityQuantumKernel(feature_map=fmap, fidelity=fidelity, enforce_psd=True)
    return FidelityQuantumKernel(feature_map=fmap, enforce_psd=True)


def run_one_seed(seed, store_predictions=False):
    """Treina clássicos + QSVMs em uma seed; retorna lista de rows (+ preds opcional)."""
    tr_s = stratified_subsample(train_full, TRAIN_SIZE, seed)
    te_s = stratified_subsample(test_full, TEST_SIZE, seed)

    pipe8 = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    X_tr8 = pipe8.fit_transform(tr_s[feat_cols])
    X_te8 = pipe8.transform(te_s[feat_cols])
    y_tr = tr_s['target'].to_numpy()
    y_te = te_s['target'].to_numpy()

    classicals = {
        'dummy': DummyClassifier(strategy='prior', random_state=seed),
        'logistic': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=seed),
        'svm_linear': CalibratedClassifierCV(
            SVC(kernel='linear', class_weight='balanced', random_state=seed), method='sigmoid', cv=3
        ),
        'svm_rbf': CalibratedClassifierCV(
            SVC(kernel='rbf', class_weight='balanced', random_state=seed), method='sigmoid', cv=3
        ),
    }

    rows = []
    preds = {}
    pred_frames = []  # long-form predictions for CSV

    for name, model in classicals.items():
        t0 = time.perf_counter()
        model.fit(X_tr8, y_tr)
        train_s = time.perf_counter() - t0
        y_hat = model.predict(X_te8)
        y_prob = proba_pos(model, X_te8)
        row = metrics_row(
            y_te, y_hat, y_prob, name,
            family='classical', n_features=8, seed=seed, training_seconds=train_s,
            positives_test=int(y_te.sum()),
        )
        rows.append(row)
        pred_frames.append(pd.DataFrame({
            'seed': seed,
            'model': name,
            'family': 'classical',
            'y_true': y_te,
            'y_hat': y_hat,
            'y_prob': y_prob if y_prob is not None else np.nan,
            'timestamp': te_s['timestamp'].to_numpy(),
        }))
        if store_predictions:
            preds[name] = {'y_true': y_te, 'y_hat': y_hat, 'y_prob': y_prob, 'family': 'classical'}

    pipe_q = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=PCA_QUBITS, random_state=seed)),
    ])
    X_tr_q = pipe_q.fit_transform(tr_s[feat_cols])
    X_te_q = pipe_q.transform(te_s[feat_cols])
    pca_explained = pipe_q.named_steps['pca'].explained_variance_ratio_.tolist()

    for qid, angular, reps in Q_VARIANTS:
        if angular == 'minmax_0_1':
            ang = MinMaxScaler(feature_range=(0.0, 1.0))
        elif angular == 'minmax_0_pi':
            ang = MinMaxScaler(feature_range=(0.0, np.pi))
        else:
            ang = None
        Xtr = ang.fit_transform(X_tr_q) if ang is not None else X_tr_q
        Xte = ang.transform(X_te_q) if ang is not None else X_te_q

        kernel = make_kernel(Xtr.shape[1], reps=reps)
        t0 = time.perf_counter()
        K_tr = kernel.evaluate(x_vec=Xtr)
        K_te = kernel.evaluate(x_vec=Xte, y_vec=Xtr)
        kernel_s = time.perf_counter() - t0

        clf = SVC(kernel='precomputed', class_weight='balanced', random_state=seed)
        t1 = time.perf_counter()
        clf.fit(K_tr, y_tr)
        train_s = time.perf_counter() - t1
        y_hat = clf.predict(K_te)
        y_prob = proba_pos(clf, K_te)
        row = metrics_row(
            y_te, y_hat, y_prob, qid,
            family='qsvm', n_features=PCA_QUBITS, seed=seed,
            angular_scaler=angular, reps=reps,
            kernel_seconds=kernel_s, training_seconds=train_s,
            positives_test=int(y_te.sum()),
            pca_explained=str(pca_explained),
        )
        rows.append(row)
        pred_frames.append(pd.DataFrame({
            'seed': seed,
            'model': qid,
            'family': 'qsvm',
            'y_true': y_te,
            'y_hat': y_hat,
            'y_prob': y_prob if y_prob is not None else np.nan,
            'timestamp': te_s['timestamp'].to_numpy(),
            'angular_scaler': angular,
            'reps': reps,
        }))
        if store_predictions:
            preds[qid] = {
                'y_true': y_te, 'y_hat': y_hat, 'y_prob': y_prob,
                'family': 'qsvm', 'K_train': K_tr,
            }

    pred_df = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    return rows, preds, pca_explained, tr_s, te_s, pred_df


## 6. Loop multi-seed (protocolo científico)

Gera `runs_raw.csv`, `summary_mean_std.csv` e teste de Wilcoxon (QSVM Farooq vs melhor clássico por AUPRC).

In [ ]:
from scipy.stats import wilcoxon
import json

all_rows = []
all_preds = []
predictions = {}
pca_explained = None
tr_s = te_s = None
t_global = time.perf_counter()

print(f'Iniciando {len(SEEDS)} seed(s) × {4 + len(Q_VARIANTS)} modelos…')
for i, seed in enumerate(SEEDS, 1):
    t0 = time.perf_counter()
    store = (seed == PRIMARY_SEED)
    print(f'\n=== [{i}/{len(SEEDS)}] seed={seed} ===')
    rows, preds, pca_explained, tr_s, te_s, pred_df = run_one_seed(seed, store_predictions=store)
    all_rows.extend(rows)
    all_preds.append(pred_df)
    pred_path = WORK / f'predictions_seed_{seed}.csv'
    pred_df.to_csv(pred_path, index=False)
    print(f'  salvo {pred_path.name} ({len(pred_df)} linhas)')
    if store:
        predictions = preds
    # print resumido
    df_seed = pd.DataFrame(rows).sort_values('average_precision', ascending=False)
    top = df_seed.iloc[0]
    elapsed = time.perf_counter() - t0
    print(f"  top={top['model']} AUPRC={top['average_precision']:.3f} F1={top['f1_extreme']:.3f}  ({elapsed/60:.1f} min)")

elapsed_total = time.perf_counter() - t_global
print(f'\nTempo total real: {elapsed_total/60:.1f} min')

runs_raw = pd.DataFrame(all_rows)
runs_raw.to_csv(WORK / 'runs_raw.csv', index=False)

predictions_all = pd.concat(all_preds, ignore_index=True)
predictions_all.to_csv(WORK / 'predictions_all_seeds.csv', index=False)
print('Salvo predictions_all_seeds.csv rows=', len(predictions_all))

metric_cols = [
    'accuracy', 'average_precision', 'f1_extreme', 'precision_extreme',
    'recall_extreme', 'balanced_accuracy', 'auroc', 'mcc',
]
summary_rows = []
for model, g in runs_raw.groupby('model'):
    row = {'model': model, 'family': g['family'].iloc[0], 'n_seeds': len(g)}
    for m in metric_cols:
        row[f'{m}_mean'] = g[m].mean()
        row[f'{m}_std'] = g[m].std(ddof=1) if len(g) > 1 else 0.0
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).sort_values('average_precision_mean', ascending=False).reset_index(drop=True)
summary.to_csv(WORK / 'summary_mean_std.csv', index=False)

print('\n=== Resumo média ± std (ordenado por AUPRC) ===')
disp = summary.copy()
for m in ['accuracy', 'average_precision', 'f1_extreme', 'auroc']:
    disp[m] = disp.apply(lambda r: f"{r[f'{m}_mean']:.3f} ± {r[f'{m}_std']:.3f}", axis=1)
display(disp[['model', 'family', 'n_seeds', 'accuracy', 'average_precision', 'f1_extreme', 'auroc']])

# Wilcoxon: Q_farooq_pipeline vs melhor clássico (por média AUPRC)
wilcoxon_result = None
q_model = 'Q_farooq_pipeline'
classic_models = summary.loc[summary['family'] == 'classical', 'model']
if q_model in set(runs_raw['model']) and len(classic_models) and len(SEEDS) >= 2:
    best_classic = (
        summary.loc[summary['family'] == 'classical']
        .sort_values('average_precision_mean', ascending=False)
        .iloc[0]['model']
    )
    a = runs_raw.loc[runs_raw['model'] == q_model].sort_values('seed')['average_precision'].to_numpy()
    b = runs_raw.loc[runs_raw['model'] == best_classic].sort_values('seed')['average_precision'].to_numpy()
    # parear por seed
    seeds_q = runs_raw.loc[runs_raw['model'] == q_model].sort_values('seed')['seed'].to_numpy()
    seeds_c = runs_raw.loc[runs_raw['model'] == best_classic].sort_values('seed')['seed'].to_numpy()
    assert np.array_equal(seeds_q, seeds_c)
    diff = a - b
    if np.allclose(diff, 0):
        stat, pval = 0.0, 1.0
    else:
        stat, pval = wilcoxon(a, b, alternative='two-sided', zero_method='wilcox')
    wilcoxon_result = {
        'qsvm': q_model,
        'classical': best_classic,
        'metric': 'average_precision',
        'delta_mean': float(diff.mean()),
        'n_seeds': int(len(diff)),
        'wins_qsvm': int((diff > 0).sum()),
        'wins_classical': int((diff < 0).sum()),
        'ties': int((diff == 0).sum()),
        'wilcoxon_stat': float(stat),
        'p_value': float(pval),
        'significant_0_05': bool(pval < 0.05),
    }
    print('\n=== Wilcoxon (QSVM Farooq − melhor clássico), AUPRC ===')
    print(json.dumps(wilcoxon_result, indent=2))
    (WORK / 'wilcoxon.json').write_text(json.dumps(wilcoxon_result, indent=2), encoding='utf-8')
elif len(SEEDS) < 2:
    print('\nWilcoxon omitido (precisa ≥2 seeds). Use MODE=\'medium\' ou \'large\'.')
else:
    print('\nWilcoxon omitido (modelo QSVM não encontrado).')


## 7. Painel visual (seed primária)

Figuras detalhadas usam `PRIMARY_SEED` (primeira da lista). As tabelas multi-seed acima são o resultado principal do experimento.


In [ ]:
import seaborn as sns
from sklearn.metrics import (
    PrecisionRecallDisplay, RocCurveDisplay, confusion_matrix,
)
from sklearn.model_selection import learning_curve

comparison_primary = (
    runs_raw.loc[runs_raw['seed'] == PRIMARY_SEED]
    .sort_values('average_precision', ascending=False)
    .reset_index(drop=True)
)
y_te = predictions[next(iter(predictions))]['y_true']

print(f'=== Painel visual — seed {PRIMARY_SEED} ===')
show = comparison_primary[[
    'model', 'accuracy', 'average_precision', 'f1_extreme',
    'precision_extreme', 'recall_extreme', 'auroc', 'family'
]].copy()
for c in show.select_dtypes(include=[float]).columns:
    show[c] = show[c].map(lambda x: f'{x:.3f}')
display(show)

# Barras média±std se multi-seed; senão seed única
plot_df = summary.sort_values('average_precision_mean')
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
metric_plot = [
    ('accuracy', 'Accuracy'),
    ('f1_extreme', 'F1 (extremo)'),
    ('average_precision', 'AUPRC'),
    ('auroc', 'AUROC'),
]
colors = {'classical': '#4C78A8', 'qsvm': '#F58518'}
for ax, (col, title) in zip(axes.ravel(), metric_plot):
    means = plot_df[f'{col}_mean'].to_numpy()
    stds = plot_df[f'{col}_std'].to_numpy()
    names = plot_df['model'].tolist()
    fams = plot_df['family'].tolist()
    y_pos = np.arange(len(names))
    ax.barh(y_pos, means, xerr=stds, color=[colors.get(f, '#999') for f in fams], capsize=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names)
    ax.set_xlim(0, 1.05)
    ax.set_title(f'{title} (mean±std, n={len(SEEDS)})')
fig.suptitle(f'Comparativo multi-seed — MODE={MODE}', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(PLOTS / 'metrics_mean_std.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrizes de confusão (seed primária)
model_names = list(predictions.keys())
n_m = len(model_names)
ncols = 3
nrows = int(np.ceil(n_m / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.6 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, name in zip(axes, model_names):
    pred = predictions[name]
    cm = confusion_matrix(pred['y_true'], pred['y_hat'], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Real 0', 'Real 1'])
    acc = accuracy_score(pred['y_true'], pred['y_hat'])
    f1 = f1_score(pred['y_true'], pred['y_hat'], zero_division=0)
    ax.set_title(f'{name}\nAcc={acc:.2f}  F1={f1:.2f}')
for ax in axes[n_m:]:
    ax.axis('off')
fig.suptitle(f'Matrizes de confusão — seed {PRIMARY_SEED}', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(PLOTS / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Matrizes (TN FP FN TP) ===')
for name in model_names:
    cm = confusion_matrix(predictions[name]['y_true'], predictions[name]['y_hat'], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    print(f'{name:22s}  TN={tn:3d} FP={fp:3d} FN={fn:3d} TP={tp:3d}')

In [ ]:
# Curvas PR / ROC (seed primária)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
pos_rate = float(np.mean(y_te))
axes[0].axhline(pos_rate, color='gray', ls=':', lw=1.2, label=f'chance ({pos_rate:.2f})')
axes[1].plot([0, 1], [0, 1], color='gray', ls=':', lw=1.2, label='chance')
for name, pred in predictions.items():
    if pred['y_prob'] is None:
        continue
    PrecisionRecallDisplay.from_predictions(pred['y_true'], pred['y_prob'], name=name, ax=axes[0])
    RocCurveDisplay.from_predictions(pred['y_true'], pred['y_prob'], name=name, ax=axes[1])
axes[0].set_title(f'Precision–Recall — seed {PRIMARY_SEED}')
axes[1].set_title(f'ROC — seed {PRIMARY_SEED}')
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.05)
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(PLOTS / 'pr_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Learning curve clássicos (uma vez; independente do multi-seed)
print('Learning curves (LogReg + SVM RBF)…')
pipe8 = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
X_full_tr = pipe8.fit_transform(train_full[feat_cols])
y_full_tr = train_full['target'].to_numpy()
max_lc = min(800, len(train_full))
idx_lc, _ = train_test_split(np.arange(len(train_full)), train_size=max_lc, stratify=y_full_tr, random_state=PRIMARY_SEED)
X_lc, y_lc = X_full_tr[idx_lc], y_full_tr[idx_lc]

lc_models = {
    'logistic': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=PRIMARY_SEED),
    'svm_rbf': SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=PRIMARY_SEED),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, model) in zip(axes, lc_models.items()):
    sizes, tr_sc, va_sc = learning_curve(
        model, X_lc, y_lc, train_sizes=np.linspace(0.2, 1.0, 5),
        cv=3, scoring='average_precision', n_jobs=-1, random_state=PRIMARY_SEED,
    )
    tr_m, tr_s = tr_sc.mean(1), tr_sc.std(1)
    va_m, va_s = va_sc.mean(1), va_sc.std(1)
    ax.plot(sizes, tr_m, 'o-', color='#4C78A8', label='Treino')
    ax.fill_between(sizes, tr_m-tr_s, tr_m+tr_s, alpha=0.15, color='#4C78A8')
    ax.plot(sizes, va_m, 'o-', color='#E45756', label='Validação CV')
    ax.fill_between(sizes, va_m-va_s, va_m+va_s, alpha=0.15, color='#E45756')
    ax.set_title(f'Learning curve — {name}'); ax.set_xlabel('N treino'); ax.set_ylabel('AUPRC')
    ax.set_ylim(0, 1.05); ax.legend()
fig.tight_layout()
fig.savefig(PLOTS / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Extras: PCA + kernel + limiar (seed primária)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
if pca_explained:
    axes[0].bar([f'PC{i+1}' for i in range(len(pca_explained))], pca_explained, color='#54A24B')
    axes[0].set_ylim(0, 1); axes[0].set_title('PCA — variância explicada')
qname = 'Q_farooq_pipeline'
if qname in predictions and 'K_train' in predictions[qname]:
    im = axes[1].imshow(predictions[qname]['K_train'], cmap='viridis', aspect='auto', vmin=0, vmax=1)
    axes[1].set_title(f'Kernel — {qname}'); fig.colorbar(im, ax=axes[1], fraction=0.046)
else:
    axes[1].axis('off')

best_name = comparison_primary.iloc[0]['model']
pred = predictions[best_name]
if pred['y_prob'] is not None:
    thresholds = np.linspace(0.05, 0.95, 19)
    f1s, precs, recs = [], [], []
    for t in thresholds:
        yh = (pred['y_prob'] >= t).astype(int)
        f1s.append(f1_score(pred['y_true'], yh, zero_division=0))
        precs.append(precision_score(pred['y_true'], yh, zero_division=0))
        recs.append(recall_score(pred['y_true'], yh, zero_division=0))
    axes[2].plot(thresholds, f1s, label='F1'); axes[2].plot(thresholds, precs, label='Precision')
    axes[2].plot(thresholds, recs, label='Recall'); axes[2].axvline(0.5, color='gray', ls='--')
    axes[2].set_title(f'Scores vs limiar — {best_name}'); axes[2].legend(fontsize=8); axes[2].set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(PLOTS / 'extras_pca_kernel_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Export final
meta = {
    'mode': MODE,
    'seeds': SEEDS,
    'n_seeds': len(SEEDS),
    'primary_seed': PRIMARY_SEED,
    'station': STATION,
    'features': feat_cols,
    'threshold': threshold,
    'train_size': TRAIN_SIZE,
    'val_size': VAL_SIZE,
    'test_size': TEST_SIZE,
    'pca_qubits': PCA_QUBITS,
    'q_variants': [list(v) for v in Q_VARIANTS],
    'elapsed_minutes': elapsed_total / 60.0,
    'estimated_minutes': est,
    'wilcoxon': wilcoxon_result,
    'best_mean_auprc_model': summary.iloc[0]['model'],
    'note': 'Farooq-style stats; temporal split fixed; stratified subsample varies by seed; multi-seed evaluation',
}
(WORK / 'meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
summary.to_csv(WORK / 'summary_mean_std.csv', index=False)

import zipfile
out_zip = WORK / 'farooq_style_kaggle_results.zip'
with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in [
        'runs_raw.csv', 'summary_mean_std.csv', 'meta.json', 'wilcoxon.json',
        'features_processed.csv', 'predictions_all_seeds.csv',
    ]:
        p = WORK / name
        if p.exists():
            zf.write(p, arcname=name)
    for p in sorted(WORK.glob('predictions_seed_*.csv')):
        zf.write(p, arcname=p.name)
    for p in sorted(PLOTS.glob('*.png')):
        zf.write(p, arcname=f'plots/{p.name}')

print('Salvo:', out_zip)
print(f'Tempo real: {elapsed_total/60:.1f} min | estimado: {est:.0f} min')
print('Arquivos:', [
    'features_processed.csv', 'runs_raw.csv', 'summary_mean_std.csv',
    'predictions_all_seeds.csv', 'predictions_seed_*.csv', 'wilcoxon.json', 'plots/',
])
if ON_KAGGLE:
    print('Kaggle Output → farooq_style_kaggle_results.zip')


### Como usar

1. Na célula de config, escolha:
   ```python
   MODE = 'small'    # ~1–3 min, 1 seed
   MODE = 'medium'   # ~20–40 min, 5 seeds
   MODE = 'large'    # ~6–12 h, 10 seeds
   ```
2. Settings Kaggle: Internet **ON**, Accelerator **None** (para `large`, evite cortar a sessão por inatividade)
3. Run All
4. Baixe `farooq_style_kaggle_results.zip`

**Resultados principais:** `summary_mean_std.csv` + `wilcoxon.json` (se ≥2 seeds) + `features_processed.csv` + `predictions_*.csv`.  
O painel visual detalha só a seed primária.
